# Ouvrir une connexion série

In [ ]:
import serial

# Ouvrir le port
ser = serial.Serial(
    port="COM3",        # sous Windows : COM3, COM4... / sous Linux : /dev/ttyUSB0, /dev/ttyS0
    baudrate=9600,       # vitesse de transmission (doit correspondre à l'automate !)
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_NONE,
    stopbits=serial.STOPBITS_ONE,
    timeout=1            # secondes d'attente max avant d'abandonner une lecture
)

print(ser.is_open)  # True si la connexion est bien ouverte

# Trouver le bon port automatiquement

In [ ]:
import serial.tools.list_ports

ports = serial.tools.list_ports.comports()
for p in ports:
    print(p.device, "-", p.description)

# Lire des données

In [ ]:
# Lire une ligne (jusqu'au \n)
ligne = ser.readline()
print(ligne)                          # b'25.3;OK\r\n'  -> bytes !
print(ligne.decode("utf-8").strip())  # '25.3;OK' -> texte propre

# Boucle de lecture continue

In [ ]:
import serial
import time

ser = serial.Serial("COM3", 9600, timeout=1)

try:
    while True:
        if ser.in_waiting > 0:              # des données sont arrivées
            ligne = ser.readline().decode("utf-8", errors="ignore").strip()
            if ligne:
                print(f"Reçu : {ligne}")
        time.sleep(0.1)
except KeyboardInterrupt:
    print("Arrêt demandé")
finally:
    ser.close()   # toujours fermer le port proprement

# Lire des holding Registers

In [ ]:
from pymodbus.client import ModbusSerialClient

client = ModbusSerialClient(
    port="COM3",
    baudrate=9600,
    bytesize=8,
    parity="N",
    stopbits=1,
    timeout=1
)

if client.connect():
    # Lire 2 holding registers à partir de l'adresse 0, esclave n°1
    resultat = client.read_holding_registers(address=0, count=2, slave=1)
    
    if not resultat.isError():
        print("Valeurs lues :", resultat.registers)
    else:
        print("Erreur de lecture :", resultat)
    
    client.close()
else:
    print("Connexion impossible")

# Écrire un registre

In [ ]:
client.write_register(address=1, value=100, slave=1)

# Lire des coils (bits)

In [ ]:
resultat = client.read_coils(address=0, count=8, slave=1)
print(resultat.bits)  # [True, False, True, ...]

# Boucle de lecture continue + sauvegarde (le vrai cas d'usage

In [ ]:
from pymodbus.client import ModbusSerialClient
import pandas as pd
from datetime import datetime
import time

client = ModbusSerialClient(port="COM3", baudrate=9600, timeout=1)
donnees = []

if client.connect():
    try:
        while True:
            resultat = client.read_holding_registers(address=0, count=2, slave=1)
            if not resultat.isError():
                donnees.append({
                    "timestamp": datetime.now(),
                    "temperature": resultat.registers[0] / 10,   # ex: si stocké en dixièmes
                    "pression": resultat.registers[1] / 100
                })
                print(donnees[-1])
            time.sleep(1)
    except KeyboardInterrupt:
        print("Arrêt")
    finally:
        client.close()
        df = pd.DataFrame(donnees)
        df.to_csv("mesures_modbus.csv", index=False)

# Projet
Fonction	Rôle
lister_ports()	trouve automatiquement les ports série disponibles
ouvrir_connexion()	configure et ouvre la liaison série
lire_ligne()	lit une ligne et la décode en texte
envoyer_commande()	envoie une commande à l'automate
parser_trame()	transforme le texte reçu en données exploitables (dictionnaire)
boucle_acquisition()	lit en continu, horodate, et sauvegarde en CSV

Points importants à retenir dans ce script :

if __name__ == "__main__": → c'est le point d'entrée du programme, tout ce qui est dedans ne s'exécute que si tu lances ce fichier directement (pas si tu l'importes ailleurs)
try / except / finally → structure essentielle en automatisme : même si une erreur survient ou que tu arrêtes avec Ctrl+C, le finally garantit que :
les données déjà collectées sont sauvegardées
le port série est bien refermé (sinon il reste "bloqué" pour la prochaine exécution)
parser_trame() est à adapter — j'ai supposé un format "temperature;pression;statut", mais c'est le format exact envoyé par ton automate qu'il faut mettre ici. Si tu me donnes un exemple de trame réelle, je l'ajuste.
PORT = "COM3" — à changer selon ton port réel (utilise lister_ports() pour le trouver).


In [ ]:

import serial
import serial.tools.list_ports
import pandas as pd
from datetime import datetime
import time


# ---------------------------------------------------------------------
# 1. LISTER LES PORTS SÉRIE DISPONIBLES
# ---------------------------------------------------------------------
def lister_ports():
    """Affiche tous les ports série détectés sur la machine."""
    ports = serial.tools.list_ports.comports()
    print("Ports série disponibles :")
    if not ports:
        print("  Aucun port détecté.")
    for p in ports:
        print(f"  - {p.device} : {p.description}")
    return [p.device for p in ports]


# ---------------------------------------------------------------------
# 2. OUVRIR LA CONNEXION SÉRIE
# ---------------------------------------------------------------------
def ouvrir_connexion(port, baudrate=9600, timeout=1):
    """Ouvre et retourne une connexion série configurée."""
    ser = serial.Serial(
        port=port,
        baudrate=baudrate,
        bytesize=serial.EIGHTBITS,
        parity=serial.PARITY_NONE,
        stopbits=serial.STOPBITS_ONE,
        timeout=timeout
    )
    print(f"Connexion ouverte sur {port} ({baudrate} bauds) : {ser.is_open}")
    return ser


# ---------------------------------------------------------------------
# 3. LIRE UNE LIGNE DE DONNÉES
# ---------------------------------------------------------------------
def lire_ligne(ser):
    """Lit une ligne envoyée par l'automate et la décode en texte."""
    brut = ser.readline()                              # retourne des bytes
    texte = brut.decode("utf-8", errors="ignore").strip()
    return texte


# ---------------------------------------------------------------------
# 4. ENVOYER UNE COMMANDE À L'AUTOMATE
# ---------------------------------------------------------------------
def envoyer_commande(ser, commande):
    """Envoie une commande texte à l'automate (encodée en bytes)."""
    ser.write((commande + "\n").encode("utf-8"))
    print(f"Commande envoyée : {commande}")


# ---------------------------------------------------------------------
# 5. PARSER UNE TRAME DE DONNÉES
# ---------------------------------------------------------------------
def parser_trame(ligne):
    """
    Transforme une ligne reçue en dictionnaire exploitable.
    Format attendu ici, à adapter selon ton automate : "temperature;pression;statut"
    Exemple de ligne reçue : "25.3;4.1;MARCHE"
    """
    parties = ligne.split(";")
    if len(parties) == 3:
        try:
            return {
                "timestamp": datetime.now(),
                "temperature_C": float(parties[0]),
                "pression_bar": float(parties[1]),
                "statut": parties[2]
            }
        except ValueError:
            return None
    return None


# ---------------------------------------------------------------------
# 6. BOUCLE DE LECTURE CONTINUE + SAUVEGARDE CSV
# ---------------------------------------------------------------------
def boucle_acquisition(ser, fichier_sortie="mesures_automate.csv", duree_max=None):
    """
    Lit en continu les données de l'automate et les enregistre.
    - duree_max : nombre de secondes avant arrêt automatique (None = illimité, arrêt avec Ctrl+C)
    """
    donnees = []
    debut = time.time()

    print("Acquisition en cours... (Ctrl+C pour arrêter)")
    try:
        while True:
            if ser.in_waiting > 0:                      # des données sont arrivées
                ligne = lire_ligne(ser)
                mesure = parser_trame(ligne)
                if mesure:
                    donnees.append(mesure)
                    print(mesure)

            if duree_max and (time.time() - debut) > duree_max:
                print("Durée maximale atteinte, arrêt.")
                break

            time.sleep(0.1)

    except KeyboardInterrupt:
        print("\nArrêt demandé par l'utilisateur.")

    finally:
        # Sauvegarde même si le programme est interrompu
        if donnees:
            df = pd.DataFrame(donnees)
            df.to_csv(fichier_sortie, index=False)
            print(f"{len(df)} mesures sauvegardées dans '{fichier_sortie}'")
        else:
            print("Aucune donnée collectée.")

    return donnees


# ---------------------------------------------------------------------
# PROGRAMME PRINCIPAL
# ---------------------------------------------------------------------
if __name__ == "__main__":

    # Étape 1 : voir les ports disponibles
    ports_disponibles = lister_ports()

    # Étape 2 : choisir le port (à adapter selon ton cas : "COM3" ou "/dev/ttyUSB0")
    PORT = "COM3"
    BAUDRATE = 9600

    ser = None
    try:
        # Étape 3 : ouvrir la connexion
        ser = ouvrir_connexion(PORT, BAUDRATE)

        # Étape 4 : envoyer une commande de test (optionnel selon ton automate)
        envoyer_commande(ser, "GET_STATUS")

        # Étape 5 : lancer l'acquisition continue (ici limitée à 60 secondes)
        boucle_acquisition(ser, fichier_sortie="mesures_automate.csv", duree_max=60)

    except serial.SerialException as e:
        print(f"Erreur de connexion série : {e}")

    finally:
        # Étape 6 : toujours fermer proprement le port, même en cas d'erreur
        if ser and ser.is_open:
            ser.close()
            print("Connexion série fermée.")